## Imports and definitions

In [ ]:
from ddsp import DDSP, AudioDataset
from ddsp.callbacks import BetaWarmupEpochCallback
from ddsp.synths import NoiseBandSynth, SineSynth
from ddsp.utils import find_checkpoint

from lightning.pytorch.callbacks import ModelCheckpoint
from lightning import Trainer

from IPython.display import Audio, display

from torch.utils.data import DataLoader, Subset, random_split

import torch
import umap

torch.set_float32_matmul_precision('medium')
torch.set_default_tensor_type('torch.cuda.FloatTensor')
torch.set_default_device('cuda')

import os

experiment_root = '/home/btadeusz/code/ddsp_vae/experiments/mixing_datasets/'
training_root = os.path.join(experiment_root, 'training')

In [ ]:
# Dataset parameters
chunk_duration = 2.0
sampling_rate = 44100
n_signal = chunk_duration * sampling_rate
batch_size = 8

# Model parameters
latent_size = num_params = 4
max_freq = 22050
n_filters = 500
n_sines = 100

# Training parameters
warmup_start = 300
warmup_end = 500
beta = 1.0
max_epochs = 1000
learning_rate = 1e-4


In [ ]:
def get_dataset_split(dataset_path, validation_split=0.2):
  """
  Splits the dataset into training and validation sets.
  """
  generator=torch.Generator(device='cuda')

  dataset_A = AudioDataset(dataset_path=dataset_path, n_signal=n_signal)
  total_len = len(dataset_A)

  val_len = int(validation_split * total_len)  # 20% for validation
  indices = torch.randperm(total_len, generator=generator)

  val_indices = indices[:val_len]
  train_indices = indices[val_len:]

  train_set = Subset(dataset_A, train_indices)
  val_set = Subset(dataset_A, val_indices)

  train_loader = DataLoader(train_set, batch_size, shuffle=True, num_workers=0, generator=generator)
  val_loader = DataLoader(val_set, batch_size, shuffle=False, num_workers=0, generator=generator)

  return train_loader, val_loader


def build_ddsp_model():
  """
  Builds the DDSP model with the specified configurations.
  """
  nbn = NoiseBandSynth.to_config(
    n_filters=n_filters,
    fs=sampling_rate,
  )

  sines = SineSynth.to_config(
    n_sines=n_sines,
    fs=sampling_rate,
  )

  ddsp = DDSP(
    synth_configs=[nbn, sines],
    fs=sampling_rate,
    latent_size=latent_size,
    num_params=num_params,
    learning_rate=learning_rate,
    perceptual_loss_weight=0,
    plateau_patience=200,
  ).to('cuda')

  return ddsp


def build_trainer(model_training_path):
  training_callbacks = []
  beta_warmup = BetaWarmupEpochCallback(
    beta=beta,
    start_epoch=warmup_start,
    end_epoch=warmup_end
  )
  training_callbacks.append(beta_warmup)

  best_checkpoint_callback = ModelCheckpoint(
    filename='best',
    monitor='val_loss',
    mode='min',
    save_top_k=1,
    save_last=True,
    dirpath=model_training_path
  )
  training_callbacks.append(best_checkpoint_callback)

  trainer = Trainer(
    callbacks=training_callbacks,
    max_epochs=max_epochs,
    accelerator='cuda',
    precision=16,
  )

  return trainer


def train_on_dataset(dataset_path, model_name=None):
  """
  Trains the DDSP model on a specific dataset.
  """
  train_loader, val_loader = get_dataset_split(dataset_path)

  model = build_ddsp_model()

  model_training_path = os.path.join(training_root, model_name)
  trainer = build_trainer(model_training_path)

  ckpt = find_checkpoint(model_training_path, return_none=True, typ='last')
  print("Found ckpt: ", ckpt)
  trainer.fit(model, train_loader, val_loader, ckpt_path=ckpt)

  return model, trainer, train_loader, val_loader


def examine_model(model, val_loader):
  """
  Examines the trained model by displaying original and autoencoded audio samples,
  as well as reporting on the loss.
  """
  model = model.cuda()
  model.eval()
  with torch.no_grad():
    x_audio = next(iter(val_loader))
    x_audio = x_audio.to('cuda')

    # Forward pass through the model
    y_audio = model(x_audio).squeeze(1)

    print(x_audio.shape, y_audio.shape)

    # Display original and reconstructed audio
    for j in range(x_audio.shape[0]):
      print(f"Sample {j + 1}:")
      x_audio_j = x_audio[j].cpu()
      y_audio_j = y_audio[j].cpu()

      display(Audio(x_audio_j.numpy(), rate=sampling_rate))
      display(Audio(y_audio_j.numpy(), rate=sampling_rate))


## Train model A

In [ ]:
model_A, trainer_A, train_loader_A, val_loader_A = train_on_dataset('/mnt/mariadata/datasets/syntex/DS_Pistons_1.1/audio', model_name=f'model_A_beta_{beta}')

### Results

In [ ]:
examine_model(model_A, val_loader_A)

## Train model B

In [ ]:
model_B, trainer_B, train_loader_B, val_loader_B = train_on_dataset('/mnt/mariadata/datasets/syntex/DS_BasicFM_1.1/audio', model_name=f'model_B_beta_{beta}')

In [ ]:
examine_model(model_B, val_loader_B)

### Results

## Train model A+B

In [ ]:
model_AB, trained_AB, train_loader_AB, val_loader_AB = train_on_dataset('/mnt/mariadata/datasets/syntex/DS_*/audio', model_name=f'model_AB_beta_{beta}')

In [ ]:
examine_model(model_AB, val_loader_AB)

# Plots

In [ ]:
import numpy as np
from sklearn.decomposition import PCA
import torch
import seaborn as sns

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

# Helper to encode a batch into latent space
def encode_latents(model, loader, n_batches=2):
  model.eval()
  model.streaming(False)
  model = model.cuda()
  latents = []
  with torch.no_grad():
    for i, batch in enumerate(loader):
      batch = batch.to('cuda')
      # Get mean of latent distribution (mu)
      mu, logvar = model.encoder(batch)
      z, _ = model.encoder.reparametrize(mu, logvar)
      # z = logvar
      # z = mu
      latents.append(z.cpu())
      if i + 1 >= n_batches:
        break
  return torch.concatenate(latents, axis=0)

def compute_losses(model, loader, n_batches=1000):
  model.eval()
  model = model.cuda()
  total_recons_loss = 0.0
  total_kld = 0.0
  total_samples = 0
  with torch.no_grad():
    for i, batch in enumerate(loader):
      batch = batch.to('cuda')
      _, losses = model._autoencode(batch)
      recons_loss = losses['recons_loss'].item()
      kld = losses['kld'].item()
      total_recons_loss += recons_loss * batch.size(0)
      total_kld += kld * batch.size(0)
      total_samples += batch.size(0)
      if i + 1 >= n_batches:
        break
  mean_recons_loss = total_recons_loss / total_samples
  mean_kld = total_kld / total_samples
  return mean_recons_loss, mean_kld

In [ ]:
from matplotlib.lines import Line2D

def plot_model_space(model_name: str, dataset_loaders: list, dataset_names: list, colors: list = None, n_batches: int = 2, figure_path: str = None):
  """
  Plots the latent space of the model using UMAP and PCA.
  """
  num_datasets = len(dataset_loaders)

  # Load model
  model_training_path = os.path.join(training_root, model_name)
  ckpt = find_checkpoint(model_training_path, return_none=False, typ='last')
  model = DDSP.load_from_checkpoint(ckpt)

  # Encode samples and compute losses per dataset
  latents = []
  mrstft_losses = []
  klds = []
  for loader in dataset_loaders:
    cur_latents = encode_latents(model, loader, n_batches)

    # Extract bin to bin
    cur_latents_bins = cur_latents.reshape(-1, cur_latents.shape[-1])
    latents.append(cur_latents_bins)

    # Compute losses
    cur_mrstft_loss, cur_kld = compute_losses(model, loader)
    mrstft_losses.append(cur_mrstft_loss)
    klds.append(cur_kld)


  # Stack and create labels
  X = np.vstack(latents)
  y = np.array(sum([[let]*len(dataset_latents) for let, dataset_latents in zip(dataset_names, latents)], []))

  # 2d UMAP projection
  umap_model = umap.UMAP(n_components=2, random_state=42)
  X_umap = umap_model.fit_transform(X)

  # Plot the space
  fig, ax = plt.subplots(figsize=(10, 8), dpi=150)

  # 5 colors
  if colors is None:
    colors = sns.color_palette("husl", n_colors=num_datasets)

  for label, color in zip(dataset_names[:num_datasets], colors):
    idx = y == label
    sns.scatterplot(
      x=X_umap[idx, 0], y=X_umap[idx, 1],
      label=f'Dataset {label}', alpha=0.5, s=2, marker='o', color=color, ax=ax, edgecolor=None
    )

  ax.set_title(f'z (UMAP), β={beta}', fontsize=14, pad=35)

  subtitle = "\n".join([
    ', '.join([f'KLD ({let}): {kld:.3f}' for let, kld in zip(dataset_names[:num_datasets], klds)]),
    ', '.join([f'recons_loss ({name}): {mrstft:.3f}' for name, mrstft in zip(dataset_names[:num_datasets], mrstft_losses)])
  ])

  ax.text(0.5, 1.01, subtitle, fontsize=11, ha='center', va='bottom', transform=ax.transAxes)

  # Remove axes
  ax.set_xlabel('')
  ax.set_ylabel('')
  ax.set_xticks([])
  ax.set_yticks([])
  for spine in ax.spines.values():
      spine.set_visible(False)

  # Custom handles for larger dots in legend
  handles = [
      Line2D([0], [0], marker='o', color='w', label=f'{name}',
            markerfacecolor=color, markersize=8) for name, color in zip(dataset_names[:num_datasets], colors)
  ]
  ax.legend(handles=handles, scatterpoints=1, loc='best', frameon=True)

  # Remove grid
  ax.grid(False)

  plt.tight_layout()
  if figure_path is not None:
    plt.savefig(figure_path, bbox_inches='tight', dpi=150)
  else:
    plt.show()


In [ ]:
beta_values = [0.001, 0.01, 0.1, 1.0]
colors = sns.color_palette("dark", n_colors=2)
plots_root = os.path.join(experiment_root, 'plots')

for beta in beta_values:
  print(f"Plotting model space for beta={beta}")
  # model A
  plot_model_space(f'model_A_beta_{beta}', [val_loader_A], dataset_names=['Pistons'], colors=colors, figure_path=os.path.join(plots_root, f'model_A_beta_{beta}.png'))

  # model B
  plot_model_space(f'model_B_beta_{beta}', [val_loader_B], dataset_names=['BasicFM'], colors=colors[1:], figure_path=os.path.join(plots_root, f'model_B_beta_{beta}.png'))

  # model A + B
  plot_model_space(f'model_AB_beta_{beta}', [val_loader_A, val_loader_B], dataset_names=['Pistons', 'BasicFM'], colors=colors, figure_path=os.path.join(plots_root, f'model_AB_beta_{beta}.png'))
